# Visualize Depth Predictions
Quick check of the generated depth maps from `generate_amazon_depth.py`.

In [ ]:
import os
import cv2
import numpy as np
import pickle
import lz4.block
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 5)

In [ ]:
PKLZ_DIR = './datasets/amazon_data/test_data'
DEPTH_DIR = './datasets/amazon_data_depth/test_data/depth'

# Get the 3 generated depth files
depth_files = sorted([f for f in os.listdir(DEPTH_DIR) if f.endswith('.png')])
print(f'Found {len(depth_files)} depth maps')
for f in depth_files:
    print(f'  {f}')

In [ ]:
for depth_fname in depth_files:
    # Load depth (16-bit PNG, values in mm)
    depth = cv2.imread(os.path.join(DEPTH_DIR, depth_fname), cv2.IMREAD_UNCHANGED).astype(np.float32)

    # Load corresponding original image
    pklz_fname = depth_fname.replace('_depth.png', '.pklz')
    pklz_path = os.path.join(PKLZ_DIR, pklz_fname)
    with open(pklz_path, 'rb') as fp:
        rec = pickle.loads(lz4.block.decompress(fp.read()))
    rgb = np.array(Image.open(BytesIO(rec['image_data'])))
    dims = rec['dimensions']
    weight = rec.get('weight', 'N/A')

    # Resize+pad RGB to match depth dimensions
    border = (640 - 480) // 2
    rgb_resized = cv2.resize(rgb, (480, 480))
    rgb_padded = cv2.copyMakeBorder(rgb_resized, 0, 0, border, border,
                                     cv2.BORDER_CONSTANT, value=[255, 255, 255])

    # Plot: original | processed | depth
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].imshow(rgb)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(rgb_padded)
    axes[1].set_title('Processed (480x640)')
    axes[1].axis('off')

    depth_masked = np.ma.masked_where(depth == 0, depth)
    im = axes[2].imshow(depth_masked, cmap='rainbow')
    axes[2].set_title('Predicted Depth (mm)')
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    name = pklz_fname.replace('.pklz', '')
    dims_str = ' x '.join(dims) + ' in'
    nz = depth[depth > 0]
    fig.suptitle(f'{name}\nDims: {dims_str} | Weight: {weight} lbs | '
                 f'Depth range: {nz.min():.0f}-{nz.max():.0f} mm',
                 fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()